# 00 — Data Check

Purpose: inspect the raw `scraped_news` table (3000 rows) before writing any cleaning/date-processing code.
This notebook answers: what do the real date formats look like, how many rows have quality issues, and how many likely duplicates exist — so `cleaning.py` and `date_processing.py` are built against reality, not guesses.

In [2]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
from sqlalchemy import text
import database as db

pd.set_option('display.max_colwidth', 100)

## 1. Load all raw rows into a dataframe

In [4]:
with db.get_connection() as conn:
    df = pd.read_sql(text('SELECT id, title, content, date, link FROM scraped_news'), conn)

print(f'Total rows: {len(df)}')
df.head()

Total rows: 16794


,id,title,content,date,link
0,1,CCID obtains 90-day detention order for drug trafficker Shiran Basik,The Central Criminal Investigation Department (CCID) has obtained approval to detain 48-year-old...,17 Aug 2026,https://www.dailymirror.lk/breaking-news/CCID-obtains-90-day-detention-order-for-drug-trafficker...
1,2,Colombo port under threat? Galathea is the warning shot,"Colombo has the port. India has the cargo. With nearly, Sri Lanka’s record-breaking port perform...",17 Aug 2026,https://www.dailymirror.lk/breaking-news/Colombo-port-under-threat-Galathea-is-the-warning-shot/...
2,3,"Sri Lanka CERT warns of scam impersonating Central Bank, police officials",Sri Lanka CERT has warned the public about a sophisticated fraud scheme in which scammers impers...,17 Aug 2026,https://www.dailymirror.lk/breaking-news/Sri-Lanka-CERT-warns-of-scam-impersonating-Central-Bank...
3,4,Does the Public Have Any Say in Sri Lanka's Budget? The Score Is 9 Out of 100,"Sri Lanka is making notable progress in publishing fiscal data, yet ordinary citizens remain alm...",17 Aug 2026,https://www.dailymirror.lk/breaking-news/Does-the-Public-Have-Any-Say-in-Sri-Lanka-s-Budget-The-...
4,5,Trump orders Pentagon to cut back military exercises with South Korea,U.S. President Donald Trump on Sunday instructed the Pentagon to ’’substantially reduce’’ ​joint...,17 Aug 2026,https://www.dailymirror.lk/breaking-news/Trump-orders-Pentagon-to-cut-back-military-exercises-wi...


## 2. Basic null / empty checks

In [5]:
checks = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'empty_string_count': (df.astype(str).apply(lambda col: (col.str.strip() == '').sum()))
})
checks

,null_count,empty_string_count
id,0,0
title,0,0
content,0,0
date,0,0
link,0,0


## 3. Content length distribution

Flags rows likely too short to be useful (matches `MIN_CONTENT_LENGTH_CHARS` in `config.py`).

In [6]:
df['content_len_chars'] = df['content'].astype(str).str.len()
print(df['content_len_chars'].describe())

import config
short = df[df['content_len_chars'] < config.MIN_CONTENT_LENGTH_CHARS]
print(f"\nRows below MIN_CONTENT_LENGTH_CHARS ({config.MIN_CONTENT_LENGTH_CHARS} chars): {len(short)}")
short[['id', 'title', 'content_len_chars']].head(20)

count    16794.000000
mean       206.056806
std         63.301414
min          3.000000
25%        161.000000
50%        198.000000
75%        242.000000
max        450.000000
Name: content_len_chars, dtype: float64

Rows below MIN_CONTENT_LENGTH_CHARS (50 chars): 2


,id,title,content_len_chars
2727,2728,Sri Lanka Schools Dialog Under-19 League Rugby Tournament 2026,40
6353,6354,Shanaka Rewrites Record Books with 19-Ball Fifty,3


## 4. Distinct date value patterns

This is the most important check for `date_processing.py` — we need every distinct *shape* of date value present, not just a few examples.

In [7]:
print(f"Distinct raw date values: {df['date'].nunique()} (out of {len(df)} rows)")
df['date'].value_counts().head(30)

Distinct raw date values: 571 (out of 16794 rows)


date
27 Nov 2025    76
08 Jul 2026    75
29 Nov 2025    72
28 Nov 2025    63
02 Jul 2026    62
09 Apr 2026    61
19 May 2026    60
08 Apr 2026    59
05 Aug 2026    58
25 Jun 2026    57
01 Dec 2025    57
05 Dec 2025    56
07 Aug 2026    55
17 Mar 2026    55
03 Dec 2025    55
16 Jun 2026    55
09 Jun 2026    55
18 Jun 2026    54
06 Aug 2026    54
25 Mar 2026    53
10 Sep 2025    53
01 Jul 2026    52
07 Jul 2026    52
17 Jun 2026    52
19 Dec 2025    52
11 Nov 2025    51
30 Sep 2025    51
04 Aug 2026    51
26 May 2026    51
02 Dec 2025    50
Name: count, dtype: int64

In [8]:
# Group by rough pattern (letters vs digits vs known keywords) to see the SHAPES of date values,
# not just the top literal values (which will be dominated by '1 hours ago' etc.)
import re

def date_pattern(val):
    s = str(val).strip()
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}.*', s):
        return 'ISO_DATE (YYYY-MM-DD...)'
    if re.fullmatch(r'\d+\s+hours?\s+ago', s, re.IGNORECASE):
        return 'N hours ago'
    if re.fullmatch(r'\d+\s+days?\s+ago', s, re.IGNORECASE):
        return 'N days ago'
    if re.fullmatch(r'\d+\s+minutes?\s+ago', s, re.IGNORECASE):
        return 'N minutes ago'
    if s.lower() in ('yesterday', 'today'):
        return 'Yesterday/Today keyword'
    if re.fullmatch(r'[A-Za-z]+\s+\d{1,2},?\s+\d{4}', s):
        return 'Month DD, YYYY'
    if s == '' or s.lower() == 'none' or s.lower() == 'nan':
        return 'EMPTY/NULL'
    return 'OTHER/UNRECOGNIZED'

df['date_pattern'] = df['date'].apply(date_pattern)
pattern_counts = df['date_pattern'].value_counts()
print(pattern_counts)
print(f"\nTotal patterns found: {len(pattern_counts)}")

date_pattern
OTHER/UNRECOGNIZED    16794
Name: count, dtype: int64

Total patterns found: 1


In [9]:
# Show a few real examples from each pattern bucket, especially OTHER/UNRECOGNIZED —
# these are the values date_processing.py needs explicit handling for.
for pattern in df['date_pattern'].unique():
    examples = df[df['date_pattern'] == pattern]['date'].unique()[:5]
    print(f"--- {pattern} ---")
    for ex in examples:
        print(f"  {ex!r}")
    print()

--- OTHER/UNRECOGNIZED ---
  '17 Aug 2026'
  '16 Aug 2026'
  '15 Aug 2026'
  '14 Aug 2026'
  '13 Aug 2026'



## 5. Duplicate detection preview

Exact duplicates by `link` and by `title` — a first pass before the real `deduplication.py` (which will also check content-hash and near-duplicate similarity).

In [10]:
dup_links = df[df.duplicated(subset=['link'], keep=False)].sort_values('link')
print(f"Rows sharing a duplicate link: {len(dup_links)}")
dup_links[['id', 'title', 'link']].head(20)

Rows sharing a duplicate link: 0


,id,title,link


In [11]:
dup_titles = df[df.duplicated(subset=['title'], keep=False)].sort_values('title')
print(f"Rows sharing an exact duplicate title: {len(dup_titles)}")
dup_titles[['id', 'title', 'date']].head(20)

Rows sharing an exact duplicate title: 87


,id,title,date
13913,13914,2023 Grade five scholarship exam: Cut-off marks released,02 Feb 2024
15577,15578,2023 Grade five scholarship exam: Cut-off marks released,17 Nov 2023
16246,16247,A 15-hour water cut in Colombo,21 Oct 2023
13738,13739,A 15-hour water cut in Colombo,09 Feb 2024
13581,13582,Ambitious plans underway to make SL a ‘happening’ tourist destination,15 Feb 2024
13560,13561,Ambitious plans underway to make SL a ‘happening’ tourist destination,16 Feb 2024
13491,13492,"Apply directly to respective schools, excluding grades 1, 5, 6, and A/L",19 Feb 2024
14150,14151,"Apply directly to respective schools, excluding grades 1, 5, 6, and A/L",23 Jan 2024
15393,15394,CBSL further reduces policy interest rates,24 Nov 2023
16622,16623,CBSL further reduces policy interest rates,05 Oct 2023


## 6. Summary

Fill this in after running the cells above — this becomes the spec for `cleaning.py` and `date_processing.py`.

In [12]:
print('=== SUMMARY ===')
print(f"Total rows: {len(df)}")
print(f"Rows with null/empty content: {(df['content'].astype(str).str.strip() == '').sum()}")
print(f"Rows below min content length: {len(short)}")
print(f"Distinct date patterns found: {len(pattern_counts)}")
print(f"Rows with duplicate links: {len(dup_links)}")
print(f"Rows with duplicate titles: {len(dup_titles)}")
print()
print('Date patterns breakdown:')
print(pattern_counts)

=== SUMMARY ===
Total rows: 16794
Rows with null/empty content: 0
Rows below min content length: 2
Distinct date patterns found: 1
Rows with duplicate links: 0
Rows with duplicate titles: 87

Date patterns breakdown:
date_pattern
OTHER/UNRECOGNIZED    16794
Name: count, dtype: int64


In [13]:
for val in df['date'].head(20):
    print(repr(val))

'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'
'17 Aug 2026'


In [14]:
print(f"Distinct actual date values: {df['date'].nunique()}")
df['date'].value_counts().head(20)

Distinct actual date values: 571


date
27 Nov 2025    76
08 Jul 2026    75
29 Nov 2025    72
28 Nov 2025    63
02 Jul 2026    62
09 Apr 2026    61
19 May 2026    60
08 Apr 2026    59
05 Aug 2026    58
25 Jun 2026    57
01 Dec 2025    57
05 Dec 2025    56
07 Aug 2026    55
17 Mar 2026    55
03 Dec 2025    55
16 Jun 2026    55
09 Jun 2026    55
18 Jun 2026    54
06 Aug 2026    54
25 Mar 2026    53
Name: count, dtype: int64

In [15]:
print(f"Total rows: {len(df)}")
print(f"Distinct date values: {df['date'].nunique()}")

parsed = pd.to_datetime(df['date'], format='%d %b %Y')
print(f"Date range: {parsed.min().date()} to {parsed.max().date()}")
print(f"Span: {(parsed.max() - parsed.min()).days} days")
print()

# re-check data quality on the full refreshed set
print(f"Null/empty content: {(df['content'].astype(str).str.strip() == '').sum()}")
print(f"Duplicate links: {df.duplicated(subset=['link'], keep=False).sum()}")
print(f"Duplicate titles: {df.duplicated(subset=['title'], keep=False).sum()}")
print()

# articles per week, to sanity-check there are no big gaps
weekly = parsed.dt.to_period('W').value_counts().sort_index()
print("Weeks with fewest articles (checking for gaps):")
print(weekly.sort_values().head(10))

Total rows: 16794
Distinct date values: 571
Date range: 2023-09-28 to 2026-08-17
Span: 1054 days

Null/empty content: 0
Duplicate links: 0
Duplicate titles: 87

Weeks with fewest articles (checking for gaps):
date
2026-08-17/2026-08-23     27
2025-07-21/2025-07-27     40
2023-09-25/2023-10-01     68
2023-12-25/2023-12-31     86
2025-10-20/2025-10-26     86
2024-04-22/2024-04-28     88
2024-01-08/2024-01-14    120
2025-11-03/2025-11-09    125
2024-04-01/2024-04-07    130
2024-03-25/2024-03-31    139
Freq: W-SUN, Name: count, dtype: int64


In [16]:
# check every distinct date value actually matches DD Mon YYYY — flag anything that doesn't
import re
pattern = re.compile(r'^\d{2} [A-Za-z]{3} \d{4}$')
bad_formats = df[~df['date'].astype(str).str.match(pattern)]
print(f"Rows NOT matching 'DD Mon YYYY': {len(bad_formats)}")
if len(bad_formats) > 0:
    print(bad_formats['date'].unique()[:20])

Rows NOT matching 'DD Mon YYYY': 0
